# Reproducing Gaddy & Klein silent speech recognition (open vocabulary)

Runs in **Google Colab with a GPU** (Runtime → Change runtime type → GPU). Steps:

1. Authors' code + submodules (phoneme alignments)
2. EMG dataset from Zenodo 4064409 (`emg_data.tar.gz`, 3.9 GB, CC BY 4.0)
3. Pre-trained speech-to-text weights from Zenodo 7183877 (README: WER ≈ 28 %)
4. Evaluate on the 100-utterance silent test set — **no training**
5. Count the vocabulary (how many distinct words the dataset contains)
6. Simulate our hardware: band-limit EMG to 500 Hz sampling (Nyquist 250 Hz, our recording setting) and to 250 Hz, and re-evaluate
7. (Optional) time one training epoch at 500 Hz to estimate full retraining cost

Sources: github.com/dgaddy/silent_speech · doi:10.5281/zenodo.4064409 · doi:10.5281/zenodo.7183877

In [ ]:
!nvidia-smi -L || echo 'No GPU: switch the Colab runtime to GPU'
%cd /content
!git clone https://github.com/dgaddy/silent_speech.git
%cd /content/silent_speech
!git submodule init && git submodule update
!tar -xzf text_alignments/text_alignments.tar.gz
!git rev-parse HEAD

## 1. Dependencies
Colab already has PyTorch. `ctcdecode` is compiled from source (a few minutes); if it fails, see the note at the end.

In [ ]:
!pip -q install absl-py librosa soundfile unidecode jiwer==2.2.1 praat-textgrids noisereduce
!pip -q install git+https://github.com/parlance/ctcdecode.git
# KenLM language model used by the authors' CTC decoder (DeepSpeech 0.6.1 release)
!curl -L -o lm.binary https://github.com/mozilla/DeepSpeech/releases/download/v0.6.1/lm.binary
!ls -la lm.binary

## 2. EMG dataset (3.9 GB)

In [ ]:
!curl -L -o emg_data.tar.gz "https://zenodo.org/records/4064409/files/emg_data.tar.gz?download=1"
!echo "7f97d2182b896652999b1b2d0c69fd7b  emg_data.tar.gz" | md5sum -c -
!tar -xzf emg_data.tar.gz && rm emg_data.tar.gz
!ls emg_data

## 3. Pre-trained recognition weights
File names are read from the Zenodo API, so nothing is hard-coded.

In [ ]:
import json, subprocess, urllib.request, os
rec = json.load(urllib.request.urlopen('https://zenodo.org/api/records/7183877'))
for f in rec['files']:
    name, url = f['key'], f['links']['self']
    print(name, round(f['size']/1e6, 1), 'MB')
    subprocess.run(['curl', '-L', '-o', name, url], check=True)
    if name.endswith(('.zip', '.tar.gz', '.tgz')):
        subprocess.run(['unzip', '-o', name] if name.endswith('.zip') else ['tar', '-xzf', name], check=True)
MODEL = subprocess.run("find . -name '*.pt' -not -path './hifi_gan/*' | head -1", shell=True,
                       capture_output=True, text=True).stdout.strip()
print('model file:', MODEL)

## 4. Evaluate on the silent test set (no training)
Expected: WER around 28 % (authors' README).

In [ ]:
!python recognition_model.py --evaluate_saved "$MODEL" --output_directory eval_full 2>&1 | tail -5
!grep -i wer eval_full/log.txt | tail -3

## 5. How many distinct words are in the dataset?

In [ ]:
import glob, re, collections, json
counts = {}
for d in sorted(glob.glob('emg_data/*')):
    words = collections.Counter()
    for f in glob.glob(f'{d}/**/*_info.json', recursive=True):
        t = json.load(open(f)).get('text', '')
        words.update(re.findall(r"[a-z']+", t.lower()))
    counts[d] = words
    print(f'{d:40s} tokens={sum(words.values()):>8d}  distinct words={len(words):>6d}')
allw = sum(counts.values(), collections.Counter())
print('ALL', 'tokens', sum(allw.values()), 'distinct words', len(allw))
print('20 most common:', allw.most_common(20))

## 6. Simulate our hardware (500 Hz and 250 Hz sampling)
Our amplifier records at 500 Hz (also 250 Hz). The authors' loader assumes 1000 Hz. We keep their pipeline and remove everything above the simulated Nyquist frequency by resampling 1000 → fs → 1000 Hz right after loading, controlled by the environment variable `EMG_SIM_FS`.
First evaluate the **same pre-trained model** on band-limited input (cost of our hardware without retraining).

In [ ]:
import re
src = open('read_emg.py').read()
anchor = "    x = np.concatenate([raw_emg_before, raw_emg, raw_emg_after], 0)\n"
patch = anchor + '''    if os.environ.get('EMG_SIM_FS'):
        from scipy.signal import resample_poly
        sim = int(os.environ['EMG_SIM_FS'])
        n = x.shape[0]
        x = resample_poly(resample_poly(x, sim, 1000, axis=0), 1000, sim, axis=0)[:n]
'''
if 'EMG_SIM_FS' not in src:
    assert anchor in src, 'read_emg.py changed upstream: patch by hand'
    open('read_emg.py', 'w').write(src.replace(anchor, patch))
print('patched')

In [ ]:
!EMG_SIM_FS=500 python recognition_model.py --evaluate_saved "$MODEL" --output_directory eval_500 2>&1 | tail -5
!grep -i wer eval_500/log.txt | tail -3
!EMG_SIM_FS=250 python recognition_model.py --evaluate_saved "$MODEL" --output_directory eval_250 2>&1 | tail -5
!grep -i wer eval_250/log.txt | tail -3

## 7. (Optional) Time one training epoch at 500 Hz
Full training is 200 epochs (`train_model(..., n_epochs=200)`). This runs for 45 minutes, then stops; each finished epoch is logged with its WER. Multiply the epoch time by 200 for the full cost.

In [ ]:
!EMG_SIM_FS=500 timeout 45m python recognition_model.py --output_directory train_500 2>&1 | tail -3
!grep -n "finished epoch" train_500/log.txt | head

## Record the results
| Step | Setting | WER |
|---|---|---|
| 4 | pre-trained, 1000 Hz | |
| 6 | pre-trained, band-limited to 500 Hz (our setting) | |
| 6 | pre-trained, band-limited to 250 Hz | |
| 7 | epoch time at 500 Hz | |

Paste the numbers into `docs/decisions.md` in the team repository.

**If `ctcdecode` fails to build:** install `pyctcdecode` + `kenlm` instead and replace the `CTCBeamDecoder` call in `recognition_model.py` (`alpha=1.5, beta=1.85` as in the original); note it as a deviation, because decoding differences change WER.